# AI Resume Intelligence & Career Recommendation System

## Notebook 01 — Internal Dataset Creation

### Objective

Create an internal candidate–job matching dataset for training
machine learning models.

The final application will require only:

- Candidate Resume
- Job Description

The training dataset is an internal development artifact and will
not be supplied by the end user.

In [3]:
import os
import re
import json
import pandas as pd
import numpy as np

from pathlib import Path


In [4]:
# PROJECT PATHS

# Current notebook directory
NOTEBOOK_DIR = Path.cwd()

# Project root
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Project-level data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
ROLE_RADAR_DIR = RAW_DIR / "role_radar"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"

# Project-level model directory
MODELS_DIR = PROJECT_ROOT / "models"

# CREATE DIRECTORIES

ROLE_RADAR_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# DISPLAY PATHS

print("PROJECT PATH CONFIGURATION")

print("Project root      :", PROJECT_ROOT)
print("Notebook directory:", NOTEBOOK_DIR)
print("Data directory    :", DATA_DIR)
print("Raw directory     :", RAW_DIR)
print("Role Radar        :", ROLE_RADAR_DIR)
print("Processed         :", PROCESSED_DIR)
print("Training          :", TRAINING_DIR)
print("Models            :", MODELS_DIR)


PROJECT PATH CONFIGURATION
Project root      : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence
Notebook directory: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\notebooks
Data directory    : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data
Raw directory     : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw
Role Radar        : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar
Processed         : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\processed
Training          : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\training
Models            : c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\models


In [5]:
# DATASET INFORMATION

DATASET_NAME = "Role Radar - Job-Profile Matching Dataset"

DATASET_REPO = "jimyones/role-radar-dataset"

DATASET_SOURCE = (
    "Hugging Face Dataset Repository"
)

print("DATASET INFORMATION")

print("Dataset name :", DATASET_NAME)
print("Repository   :", DATASET_REPO)
print("Source       :", DATASET_SOURCE)


DATASET INFORMATION
Dataset name : Role Radar - Job-Profile Matching Dataset
Repository   : jimyones/role-radar-dataset
Source       : Hugging Face Dataset Repository


In [6]:
# HUGGING FACE DATASET DOWNLOADER

from huggingface_hub import hf_hub_download

print("huggingface_hub imported successfully.")


huggingface_hub imported successfully.


In [7]:
# DATASET FILES

REPO_ID = "jimyones/role-radar-dataset"

FILES_TO_DOWNLOAD = [
    "scraped_jobs.json",
    "synthetic_profiles.json",
    "phase3_pairs.json",
    "phase3_labels.json",
    "gold_labels.json"
]

print("Files to download:\n")

for file in FILES_TO_DOWNLOAD:
    print("-", file)


Files to download:

- scraped_jobs.json
- synthetic_profiles.json
- phase3_pairs.json
- phase3_labels.json
- gold_labels.json


In [8]:
# DOWNLOAD RAW ROLE RADAR DATA

import shutil

downloaded_files = {}

for filename in FILES_TO_DOWNLOAD:

    print(f"\nDownloading: {filename}")

    # Download from Hugging Face cache
    cached_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )

    # Destination inside our project
    destination_path = ROLE_RADAR_DIR / filename

    # Copy into project
    shutil.copy2(
        cached_path,
        destination_path
    )

    downloaded_files[filename] = destination_path

    print("Saved to:", destination_path)

print("DOWNLOAD COMPLETE")

for filename, path in downloaded_files.items():
    print(f"✓ {filename}")
    print(f"  {path}")



Downloading: scraped_jobs.json


Saved to: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\scraped_jobs.json

Downloading: synthetic_profiles.json
Saved to: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\synthetic_profiles.json

Downloading: phase3_pairs.json
Saved to: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\phase3_pairs.json

Downloading: phase3_labels.json
Saved to: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\phase3_labels.json

Downloading: gold_labels.json
Saved to: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\gold_labels.json
DOWNLOAD COMPLETE
✓ scraped_jobs.json
  c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\scraped_jobs.json
✓ synthetic_profiles.json
  c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\dat

In [9]:
# VERIFY RAW FILES

print("RAW DATA FILE CHECK")

all_files_exist = True

for filename in FILES_TO_DOWNLOAD:

    file_path = ROLE_RADAR_DIR / filename

    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024

        print(
            f"✓ {filename:<25} "
            f"{size_kb:,.2f} KB"
        )
    else:
        print(f"✗ {filename} NOT FOUND")
        all_files_exist = False

print("\nAll required files available:", all_files_exist)


RAW DATA FILE CHECK
✓ scraped_jobs.json         9,685.15 KB
✓ synthetic_profiles.json   417.89 KB
✓ phase3_pairs.json         1,332.96 KB
✓ phase3_labels.json        8,623.62 KB
✓ gold_labels.json          37.31 KB

All required files available: True


In [10]:
# JSON LOADER

def load_json(filename):
    # Load a JSON file from the project raw-data directory.

    file_path = ROLE_RADAR_DIR / filename

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(file)

print("JSON loader created successfully.")


JSON loader created successfully.


In [11]:
# LOAD RAW JSON DATA

jobs = load_json("scraped_jobs.json")

profiles = load_json("synthetic_profiles.json")

pairs = load_json("phase3_pairs.json")

labels = load_json("phase3_labels.json")

gold_labels = load_json("gold_labels.json")

print("RAW DATASET SIZES")

print("Jobs        :", len(jobs))
print("Profiles    :", len(profiles))
print("Pairs       :", len(pairs))
print("Labels      :", len(labels))
print("Gold labels :", len(gold_labels))


RAW DATASET SIZES
Jobs        : 2500
Profiles    : 640
Pairs       : 5000
Labels      : 22465
Gold labels : 108


In [12]:
# CREATE DATAFRAMES

jobs_df = pd.DataFrame(jobs)

profiles_df = pd.DataFrame(profiles)

pairs_df = pd.DataFrame(pairs)

labels_df = pd.DataFrame(labels)

gold_labels_df = pd.DataFrame(gold_labels)

print("DATAFRAME SHAPES")

print("Jobs        :", jobs_df.shape)

print("Profiles    :", profiles_df.shape)

print("Pairs       :", pairs_df.shape)

print("Labels      :", labels_df.shape)

print("Gold labels :", gold_labels_df.shape)


DATAFRAME SHAPES
Jobs        : (2500, 16)
Profiles    : (640, 11)
Pairs       : (5000, 8)
Labels      : (22465, 11)
Gold labels : (108, 13)


In [13]:
# JOB DATASET STRUCTURE

print("JOB DATASET")

print("Shape:", jobs_df.shape)

print("\nColumns:")

for i, column in enumerate(
    jobs_df.columns,
    start=1
):
    print(f"{i:2}. {column}")


JOB DATASET
Shape: (2500, 16)

Columns:
 1. id
 2. title
 3. company
 4. location
 5. description
 6. seniority_level
 7. employment_type
 8. job_function
 9. industry
10. url
11. posted_at
12. scraped_at
13. role_family_hint
14. domain_hint
15. experience_years_hint
16. remote_hint


In [14]:
# PROFILE DATASET STRUCTURE

print("PROFILE DATASET")

print("Shape:", profiles_df.shape)

print("\nColumns:")

for i, column in enumerate(
    profiles_df.columns,
    start=1
):
    print(f"{i:2}. {column}")


PROFILE DATASET
Shape: (640, 11)

Columns:
 1. profile_id
 2. roles
 3. skills_primary
 4. skills_secondary
 5. experience_years
 6. seniority
 7. domains
 8. preferences
 9. career_intent
10. dealbreakers
11. data_origin


In [15]:
# PAIR DATASET STRUCTURE

print("PAIR DATASET")

print("Shape:", pairs_df.shape)

print("\nColumns:")

for i, column in enumerate(
    pairs_df.columns,
    start=1
):
    print(f"{i:2}. {column}")


PAIR DATASET
Shape: (5000, 8)

Columns:
 1. pair_id
 2. profile_id
 3. job_id
 4. pair_type
 5. profile_role_family
 6. job_role_family
 7. profile_seniority
 8. skill_overlap


In [16]:
# LABEL DATASET STRUCTURE

print("LABEL DATASET")

print("Shape:", labels_df.shape)

print("\nColumns:")

for i, column in enumerate(
    labels_df.columns,
    start=1
):
    print(f"{i:2}. {column}")


LABEL DATASET
Shape: (22465, 11)

Columns:
 1. pair_id
 2. skills
 3. seniority
 4. domain
 5. location
 6. composite
 7. overqualified
 8. matches
 9. gaps
10. label_source
11. confidence


In [17]:
# GOLD LABEL DATASET STRUCTURE
print("GOLD LABEL DATASET")

print("Shape:", gold_labels_df.shape)

print("\nColumns:")

for i, column in enumerate(
    gold_labels_df.columns,
    start=1
):
    print(f"{i:2}. {column}")


GOLD LABEL DATASET
Shape: (108, 13)

Columns:
 1. pair_id
 2. skills
 3. seniority
 4. domain
 5. location
 6. composite
 7. overqualified
 8. reasoning
 9. matches
10. gaps
11. job_family
12. strategy
13. review_status


In [18]:
# SAMPLE JOB

print("SAMPLE JOB")

sample_job = jobs_df.iloc[0]

for column in jobs_df.columns:

    print(f"\n{column}:")
    print(sample_job[column])


SAMPLE JOB

id:
linkedin_4404078812

title:
Software Engineer - Development

company:
Cisco

location:
Bengaluru, Karnataka, India

description:
Please note this posting is to advertise potential job opportunities. This exact role may not be open today but could open in the near future. When you apply, a Cisco representative may contact you directly if a relevant position opens.

We are currently looking for talent to join our Intersight development team as part of Cisco's Data Center Compute portfolio.

Cisco Intersight is a cloud and on-premise based system management platform for Cisco UCS (Unified Computing System), and other Data Center products. It provides actionable intelligent level of management that enables IT organizations to analyze, simplify and automate their environments. It is driven by analytics and machine learning to provide actionable intelligence for IT operations management. Intersight Virtual Appliance is the on-premises solution that runs the Intersight capabil

In [19]:
# SAMPLE CANDIDATE PROFILE

print("SAMPLE CANDIDATE PROFILE")

sample_profile = profiles_df.iloc[0]

for column in profiles_df.columns:

    print(f"\n{column}:")
    print(sample_profile[column])


SAMPLE CANDIDATE PROFILE

profile_id:
1

roles:
['Junior Backend Engineer']

skills_primary:
['Python', 'Django', 'PostgreSQL']

skills_secondary:
['Docker', 'Redis']

experience_years:
1

seniority:
Junior

domains:
['E-commerce']

preferences:
{'locations': ['Bangalore'], 'remote': False, 'min_salary': None, 'company_sizes': []}

career_intent:
Looking for early-career backend roles at product companies

dealbreakers:
[]

data_origin:
nan


In [20]:
# PAIR TYPE DISTRIBUTION

print("PAIR TYPE DISTRIBUTION")

pair_type_counts = pairs_df["pair_type"].value_counts()

print(pair_type_counts)

print("\nPercentages:")

print(
    pairs_df["pair_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


PAIR TYPE DISTRIBUTION
pair_type
hard_negative    2000
positive         1500
negative         1500
Name: count, dtype: int64

Percentages:
pair_type
hard_negative    40.0
positive         30.0
negative         30.0
Name: proportion, dtype: float64


In [21]:
# UNIQUE ID ANALYSIS

print("UNIQUE ID ANALYSIS")

print(
    "Total job records:",
    len(jobs_df)
)

print(
    "Unique job IDs:",
    jobs_df["id"].nunique()
)

print(
    "Total profiles:",
    len(profiles_df)
)

print(
    "Unique profile IDs:",
    profiles_df["profile_id"].nunique()
)

print(
    "Total pairs:",
    len(pairs_df)
)

print(
    "Unique pair IDs:",
    pairs_df["pair_id"].nunique()
)


UNIQUE ID ANALYSIS
Total job records: 2500
Unique job IDs: 2471
Total profiles: 640
Unique profile IDs: 640
Total pairs: 5000
Unique pair IDs: 5000


In [22]:
# DUPLICATE JOB ANALYSIS

duplicate_job_count = (
    jobs_df["id"].duplicated().sum()
)

print("DUPLICATE JOB ANALYSIS")

print(
    "Total job records :",
    len(jobs_df)
)

print(
    "Unique job IDs    :",
    jobs_df["id"].nunique()
)

print(
    "Duplicate records :",
    duplicate_job_count
)


DUPLICATE JOB ANALYSIS
Total job records : 2500
Unique job IDs    : 2471
Duplicate records : 29


In [23]:
# LABEL STATISTICS

print("ROLE RADAR LABEL STATISTICS")

numeric_label_columns = [
    "skills",
    "seniority",
    "domain",
    "location",
    "composite"
]

print(
    labels_df[numeric_label_columns]
    .describe()
)


ROLE RADAR LABEL STATISTICS
             skills     seniority        domain      location     composite
count  22465.000000  22465.000000  22465.000000  22465.000000  22465.000000
mean      63.011084     86.551925     72.450568     71.784287     69.579657
std       13.091422      9.281161      9.360945     11.976485      7.810306
min       50.000000     62.000000     58.000000     60.000000     56.000000
25%       55.000000     85.000000     62.000000     68.000000     63.000000
50%       55.000000     88.000000     72.000000     68.000000     67.000000
75%       70.000000     95.000000     72.000000     68.000000     74.000000
max      100.000000     95.000000     90.000000     97.000000     98.000000


In [24]:
# LABEL SOURCE ANALYSIS

print("LABEL SOURCES")

print(
    labels_df["label_source"]
    .value_counts(dropna=False)
)


LABEL SOURCES
label_source
gemini-2.5-flash-lite+static    22465
Name: count, dtype: int64


In [25]:
# LABEL CONFIDENCE

print("LABEL CONFIDENCE")

print(
    labels_df["confidence"]
    .value_counts(dropna=False)
)


LABEL CONFIDENCE
confidence
high    22465
Name: count, dtype: int64


In [26]:
# GOLD LABEL INSPECTION

print("GOLD LABELS")

print(
    gold_labels_df[
        [
            "pair_id",
            "composite",
            "reasoning",
            "review_status"
        ]
    ].head(10)
)


GOLD LABELS
                  pair_id  composite  \
0  13_linkedin_4392811970         96   
1  55_linkedin_4392811970         95   
2  21_linkedin_4398109498         92   
3  21_linkedin_4139842127         94   
4  62_linkedin_4392811970         95   
5  18_linkedin_4392811970         96   
6  45_linkedin_4259091971         93   
7   7_linkedin_4398675342         94   
8  18_linkedin_4383134067         96   
9  25_linkedin_4379080898         93   

                                           reasoning review_status  
0  Sr Backend (TS/Node/PG/GraphQL) × Fullstack @P...           NaN  
1  Mid Fullstack (TS/React/Node/PG) × Fullstack @...           NaN  
2  Staff Backend (Java/Spring/PG/Kafka) × Fullsta...           NaN  
3  Staff Backend Arch (Java/Spring/PG/Kafka) × SD...           NaN  
4  Sr Product Eng (React/Node/GraphQL/PG) × Fulls...           NaN  
5  Sr Backend (TS/Node/PG/GraphQL) × Fullstack @P...           NaN  
6  Staff Frontend (React/TS/Next) × SE @Alterest....           N

In [27]:
# GOLD LABEL OVERLAP

pair_ids = set(
    pairs_df["pair_id"]
)

gold_pair_ids = set(
    gold_labels_df["pair_id"]
)

gold_overlap = gold_pair_ids.intersection(
    pair_ids
)

gold_not_in_pairs = gold_pair_ids - pair_ids

print("GOLD LABEL OVERLAP")

print(
    "Total gold labels       :",
    len(gold_pair_ids)
)

print(
    "Gold labels in pairs    :",
    len(gold_overlap)
)

print(
    "Gold labels not in pairs:",
    len(gold_not_in_pairs)
)


GOLD LABEL OVERLAP
Total gold labels       : 108
Gold labels in pairs    : 4
Gold labels not in pairs: 104


In [28]:
# OVERLAPPING GOLD LABELS

gold_overlap_df = gold_labels_df[
    gold_labels_df["pair_id"].isin(gold_overlap)
].copy()

print("GOLD LABELS OVERLAPPING WITH PAIRS")

print(
    gold_overlap_df[
        [
            "pair_id",
            "composite",
            "job_family",
            "review_status"
        ]
    ].to_string(index=False)
)


GOLD LABELS OVERLAPPING WITH PAIRS
                pair_id  composite       job_family review_status
144_linkedin_4399214810         79              NaN           NaN
174_linkedin_4387283659         76 customer_success       pending
 14_linkedin_4278819862         92          backend       pending
 87_linkedin_4406833892         81             data       pending


In [29]:
# DATASET METADATA

dataset_metadata = {
    "dataset_name": DATASET_NAME,
    "repository": REPO_ID,
    "jobs_records": len(jobs_df),
    "unique_jobs": int(jobs_df["id"].nunique()),
    "profiles_records": len(profiles_df),
    "unique_profiles": int(profiles_df["profile_id"].nunique()),
    "pairs_records": len(pairs_df),
    "unique_pairs": int(pairs_df["pair_id"].nunique()),
    "labels_records": len(labels_df),
    "gold_labels_records": len(gold_labels_df),
    "pair_types": pairs_df["pair_type"].value_counts().to_dict(),
    "gold_labels_in_pairs": len(gold_overlap),
    "gold_labels_not_in_pairs": len(gold_not_in_pairs)
}

print("DATASET METADATA")

for key, value in dataset_metadata.items():
    print(f"{key}: {value}")


DATASET METADATA
dataset_name: Role Radar - Job-Profile Matching Dataset
repository: jimyones/role-radar-dataset
jobs_records: 2500
unique_jobs: 2471
profiles_records: 640
unique_profiles: 640
pairs_records: 5000
unique_pairs: 5000
labels_records: 22465
gold_labels_records: 108
pair_types: {'hard_negative': 2000, 'positive': 1500, 'negative': 1500}
gold_labels_in_pairs: 4
gold_labels_not_in_pairs: 104


In [30]:
# SAVE DATASET METADATA

metadata_path = ROLE_RADAR_DIR / "dataset_metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        dataset_metadata,
        file,
        indent=4,
        default=str
    )

print("Metadata saved to:")
print(metadata_path)


Metadata saved to:
c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\dataset_metadata.json


In [31]:
# FINAL NOTEBOOK 01 SUMMARY

print("01_DATASET_CREATION COMPLETE")

print("\nDataset:")
print(DATASET_NAME)

print("\nRaw records:")
print("Jobs        :", len(jobs_df))
print("Profiles    :", len(profiles_df))
print("Pairs       :", len(pairs_df))
print("Labels      :", len(labels_df))
print("Gold labels :", len(gold_labels_df))

print("\nUnique records:")
print("Jobs        :", jobs_df["id"].nunique())
print("Profiles    :", profiles_df["profile_id"].nunique())
print("Pairs      :", pairs_df["pair_id"].nunique())

print("\nPair types:")
print(pairs_df["pair_type"].value_counts())

print("\nRaw data location:")
print(ROLE_RADAR_DIR)

print("\nMetadata:")
print(metadata_path)

print("\nNext notebook:")
print("02_data_preprocessing.ipynb")


01_DATASET_CREATION COMPLETE

Dataset:
Role Radar - Job-Profile Matching Dataset

Raw records:
Jobs        : 2500
Profiles    : 640
Pairs       : 5000
Labels      : 22465
Gold labels : 108

Unique records:
Jobs        : 2471
Profiles    : 640
Pairs      : 5000

Pair types:
pair_type
hard_negative    2000
positive         1500
negative         1500
Name: count, dtype: int64

Raw data location:
c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar

Metadata:
c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar\dataset_metadata.json

Next notebook:
02_data_preprocessing.ipynb
